# 10. LangSmith Playground - 프롬프트 실험실

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. LangSmith Playground에 접근하여 System/Human 메시지를 구성할 수 있어요
2. 변수 삽입과 모델/파라미터 설정을 통해 프롬프트를 체계적으로 테스트할 수 있어요
3. Trace에서 "Open in Playground" 기능으로 실패한 요청을 재실행하고 개선할 수 있어요
4. 여러 모델을 Side-by-side로 비교하여 최적의 모델을 선택할 수 있어요
5. 데이터셋 기반으로 프롬프트를 일괄 테스트하고 결과를 분석할 수 있어요

## 사전 지식

- 01-Setup-and-Overview: LangSmith 계정 및 API Key 설정
- 02-First-Trace ~ 09-Full-RAG-Pipeline-Debug: Trace 개념 및 UI 탐색
- ChatPromptTemplate, ChatOpenAI 기본 사용법

## Playground란?

LangSmith Playground는 **브라우저 안에서 직접 프롬프트를 작성하고 실행**할 수 있는 인터랙티브 도구예요.
코드 없이도 프롬프트를 빠르게 반복 개선할 수 있어서, 프롬프트 엔지니어링의 효율을 크게 높여줘요.

```mermaid
flowchart TD
    A["프롬프트 작성<br/>System + Human"] --> B["변수 삽입<br/>텍스트 선택 → Convert"]
    B --> C["모델 설정<br/>모델 선택 + 파라미터"]
    C --> D["CMD+Enter 실행"]
    D --> E{"결과 확인"}
    E -->|"개선 필요"| F["프롬프트 수정<br/>또는 Polly 활용"]
    E -->|"만족"| G["저장 및 커밋"]
    F --> D
    G --> H["태그 지정<br/>staging / production"]
    H --> I["데이터셋 테스트<br/>Test over dataset"]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e

    class A,B input
    class C,D,F process
    class E,G output
    class H,I storage
```

### Playground의 주요 기능

| 기능 | 설명 | 단축키/위치 |
|------|------|-----------|
| 프롬프트 실행 | 현재 설정으로 LLM 호출 | `CMD+Enter` |
| 변수 삽입 | 텍스트를 변수로 변환 | 텍스트 선택 → "Convert to variable" |
| 모델 변경 | 다른 모델로 교체 | 우상단 기어 아이콘 |
| 비교 모드 | 두 설정 Side-by-side | "+" 버튼으로 컬럼 추가 |
| 데이터셋 테스트 | 여러 입력 일괄 실행 | "Test over dataset" 드롭다운 |
| Polly AI | 자연어로 프롬프트 수정 | `Cmd+I` |
| Trace에서 열기 | 실제 요청 재실행 | Trace 뷰 → "Open in Playground" |

> 🔑 **핵심 개념**: Playground는 **반복 개선(Iteration) 루프**를 위한 도구예요. 프롬프트를 수정하고 실행하는 사이클을 빠르게 돌려서 최적의 프롬프트를 찾아가는 게 핵심이에요.

## 환경 설정

In [ ]:
# ---------------------------------------------------
# 환경 변수 로드
# ---------------------------------------------------
# .env 파일에서 OPENAI_API_KEY, LANGSMITH_API_KEY 등을 로드해요
from dotenv import load_dotenv
import os

load_dotenv()
print("환경 변수 로드 완료")

In [ ]:
# ---------------------------------------------------
# LangSmith 추적 설정
# ---------------------------------------------------
import os

os.environ["LANGSMITH_PROJECT"] = "Ch10-Playground"

print(f"프로젝트: {os.environ.get('LANGSMITH_PROJECT')}")

## 1. Playground용 프롬프트 만들어보기

Playground는 UI 도구이지만, 코드로 먼저 프롬프트를 설계하면 더 체계적으로 작업할 수 있어요.
코드에서 프롬프트를 만들고 실행해본 후, Playground로 가져가서 개선하는 흐름을 살펴볼게요.

> 🎯 **강의 포인트**: 코드 → Playground 흐름은 실무에서 매우 유용해요. 개발자는 코드로 초기 프롬프트를 작성하고, Playground에서 비개발자와 함께 프롬프트를 개선할 수 있어요.

In [3]:
# ---------------------------------------------------
# 기본 프롬프트 구성
# ---------------------------------------------------
# ChatPromptTemplate: System + Human 메시지 구조
from langchain.prompts import ChatPromptTemplate

# 고객 서비스 응답 생성 프롬프트 (예시)
# Playground에서 이 구조를 그대로 재현할 거예요
customer_service_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 {company_name}의 친절한 고객 서비스 담당자예요.
다음 지침을 따라 고객에게 응답해요:
- 항상 공감하는 태도로 시작해요
- 해결책을 명확하게 제시해요
- 응답은 {max_sentences}문장 이내로 작성해요"""),
    ("human", "{customer_inquiry}")
])

print("프롬프트 구조:")
print(customer_service_prompt)

프롬프트 구조:
input_variables=['company_name', 'customer_inquiry', 'max_sentences'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['company_name', 'max_sentences'], input_types={}, partial_variables={}, template='당신은 {company_name}의 친절한 고객 서비스 담당자예요.\n다음 지침을 따라 고객에게 응답해요:\n- 항상 공감하는 태도로 시작해요\n- 해결책을 명확하게 제시해요\n- 응답은 {max_sentences}문장 이내로 작성해요'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['customer_inquiry'], input_types={}, partial_variables={}, template='{customer_inquiry}'), additional_kwargs={})]


In [4]:
# ---------------------------------------------------
# 프롬프트에 변수 채워서 실행하기
# ---------------------------------------------------
# ChatOpenAI: gpt-4o-mini 기본 모델 사용
from langchain.chat_models import ChatOpenAI

# 기본 모델: gpt-4o-mini (비용 효율, 학생 접근성)
model = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# 체인 구성: 프롬프트 → 모델
chain = customer_service_prompt | model

# 테스트 변수 입력
response = chain.invoke({
    "company_name": "TechShop",
    "max_sentences": "3",
    "customer_inquiry": "주문한 제품이 아직 도착하지 않았어요. 어떻게 해야 하나요?"
})

print("[모델 응답]")
print(response.content)

/var/folders/h6/vm46m4nx1xz4c_2phk_wz39h0000gn/T/ipykernel_46020/3975907409.py:8: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  model = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)


[모델 응답]
주문한 제품이 아직 도착하지 않아 매우 불편하셨겠어요. 주문 상태를 확인하고, 필요한 경우 재배송 요청을 도와드리겠습니다. 고객님의 주문 번호를 말씀해 주시면 더욱 신속하게 처리해드릴 수 있습니다.


## 2. Playground UI 워크플로우

코드로 만든 프롬프트를 LangSmith Playground에서 시각적으로 개선하는 방법을 알아볼게요.

### 2.1 Playground 접근 방법

1. **LangSmith 웹사이트** (smith.langchain.com) 접속
2. 왼쪽 사이드바에서 **"Playground"** 클릭
3. 빈 Playground가 열려요

### 2.2 System/Human 메시지 구성

```
[ + System 메시지 추가 ]
당신은 {company_name}의 친절한 고객 서비스 담당자예요...

[ + Human 메시지 추가 ]
{customer_inquiry}
```

> 🔑 **핵심 개념**: Playground에서 변수를 만드는 방법은 두 가지예요:
> - **텍스트를 직접 작성** 후 중괄호 `{variable_name}`으로 감싸기
> - **텍스트 선택 후** "Convert to variable" 버튼 클릭
>
> 변수는 하단 "Variables" 패널에 자동으로 나타나서 값을 입력할 수 있어요.

### 2.3 모델 및 파라미터 설정

오른쪽 상단 **기어 아이콘(⚙️)**을 클릭하면:

| 설정 항목 | 설명 | 권장값 |
|----------|------|-------|
| Model | 사용할 LLM 모델 | gpt-4o-mini |
| Temperature | 응답 다양성 (0=결정론적, 1=창의적) | 0.7 |
| Max Tokens | 최대 응답 길이 | 1024 |
| Top P | 토큰 샘플링 범위 | 기본값 유지 |

### 2.4 실행 및 결과 확인

- **`CMD+Enter`** (Mac) 또는 **`Ctrl+Enter`** (Windows)로 실행해요
- 실행 후 하단에 모델 응답이 나타나요
- 응답 옆에 **토큰 사용량**과 **실행 시간**도 함께 표시돼요

> 💡 **실무 팁**: Playground에서 실행한 모든 요청은 자동으로 LangSmith에 Trace로 저장돼요. 나중에 Trace 뷰에서 다시 확인하거나 데이터셋으로 활용할 수 있어요.

## 3. Trace에서 Playground로 - 실패 케이스 재실행

실제 운영 중 발생한 문제를 Playground에서 바로 디버깅하는 방법이에요.
이 기능이 LangSmith Playground의 가장 강력한 특징 중 하나예요.

> 🎯 **강의 포인트**: "Open in Playground" 기능은 **실제 프로덕션 요청**을 그대로 가져와서 재실행할 수 있어요. 문제가 된 프롬프트를 수정하고 즉시 테스트할 수 있어서 디버깅 시간이 크게 줄어들어요.

In [5]:
# ---------------------------------------------------
# 의도적으로 문제가 될 수 있는 요청 생성
# ---------------------------------------------------
# 이 코드를 실행하면 LangSmith에 Trace가 생성돼요
# 그 Trace를 Playground에서 재실행하는 실습을 할 거예요
from langchain.prompts import ChatPromptTemplate
from langchain.chat_models import ChatOpenAI

# 개선이 필요한 프롬프트 (모호한 지시사항)
vague_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 도움이 되는 AI예요."),
    ("human", "{question}")
])

model = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
vague_chain = vague_prompt | model

# 복잡한 질문 - 모호한 프롬프트로는 좋은 답변이 어려워요
result = vague_chain.invoke({
    "question": "Python으로 파일을 처리하는 방법을 알려줘"
})

print("[현재 응답]")
print(result.content)
print("\n👆 이 Trace를 LangSmith에서 찾아서 'Open in Playground'를 클릭해보세요!")

[현재 응답]
Python에서 파일을 처리하는 방법은 매우 간단합니다. 파일을 열고, 읽고, 쓰고, 닫는 기본적인 작업을 통해 파일을 처리할 수 있습니다. 아래는 파일 처리의 기본적인 방법을 설명하는 예제입니다.

### 1. 파일 열기

파일을 열기 위해 `open()` 함수를 사용합니다. 이 함수는 두 가지 주요 인자를 받습니다: 파일 이름과 모드(읽기, 쓰기 등).

```python
# 파일 열기
file = open('example.txt', 'r')  # 'r'은 읽기 모드
```

### 2. 파일 읽기

파일을 읽는 방법에는 여러 가지가 있습니다.

- **전체 읽기**: `read()` 메서드를 사용하여 파일의 내용을 모두 읽습니다.

```python
content = file.read()
print(content)
```

- **한 줄씩 읽기**: `readline()` 메서드를 사용하여 파일에서 한 줄씩 읽습니다.

```python
line = file.readline()
print(line)
```

- **모든 줄을 리스트로 읽기**: `readlines()` 메서드를 사용하여 파일의 모든 줄을 리스트로 읽습니다.

```python
lines = file.readlines()
for line in lines:
    print(line)
```

### 3. 파일 쓰기

파일에 내용을 쓰려면 `open()` 함수를 통해 파일을 쓰기 모드('w' 또는 'a')로 열어야 합니다.

- **쓰기 모드**: 기존 파일을 덮어씁니다.

```python
with open('example.txt', 'w') as file:
    file.write('Hello, World!\n')
```

- **추가 모드**: 기존 파일의 끝에 내용을 추가합니다.

```python
with open('example.txt', 'a') as file:
    file.write('This is an additional line.\n')
```

### 

### Trace에서 Playground로 이동하는 방법

1. LangSmith 웹사이트에서 **Projects** → **LangSmith-Part4-Playground** 클릭
2. 방금 실행된 Trace를 클릭해요
3. 오른쪽 상단 **"Open in Playground"** 버튼 클릭
4. Playground가 해당 요청의 프롬프트와 변수로 미리 채워져 있어요!

이제 System 메시지를 더 구체적으로 수정해볼게요:

**수정 전:**
```
당신은 도움이 되는 AI예요.
```

**수정 후 (Playground에서 직접 편집):**
```
당신은 Python 전문 교육자예요. 코드 예시와 함께 단계별로 설명해요.
응답에는 반드시 다음을 포함해요:
1. 개념 설명 (2-3문장)
2. 코드 예시 (실행 가능한 코드)
3. 주의사항 또는 팁
```

> ⚠️ **자주 하는 실수**: Trace에서 "Open in Playground"가 안 보인다면, LLM 호출 노드 (ChatOpenAI)를 클릭해야 해요. 상위 체인 노드가 아닌 실제 모델 호출 노드에서 이 버튼이 나타나요.

## 4. 모델 비교 - Side-by-side 테스트

같은 프롬프트로 여러 모델을 동시에 비교하면 최적의 모델을 빠르게 찾을 수 있어요.

### Playground에서 비교 모드 사용하기

1. Playground에서 프롬프트를 작성해요
2. 오른쪽 상단 **"+"** 버튼으로 새 컬럼 추가
3. 각 컬럼에서 다른 모델 선택 (예: gpt-4o-mini vs gpt-4o)
4. `CMD+Enter`로 **두 모델을 동시에** 실행
5. 응답 품질, 토큰 수, 실행 시간을 나란히 비교

> 💡 **실무 팁**: 모델 비교 시 확인할 항목들이에요:
> - **응답 품질**: 지시사항을 얼마나 잘 따르는지
> - **토큰 효율**: 같은 정보를 더 적은 토큰으로 전달하는지
> - **속도**: 실시간 사용자 경험에 중요한 지연 시간
> - **비용**: 토큰 당 비용 × 예상 사용량

코드에서도 간단히 비교를 시뮬레이션해볼게요:

In [6]:
# ---------------------------------------------------
# 코드에서 모델 비교 (Playground 비교 모드의 코드 버전)
# ---------------------------------------------------
import time
from langchain.prompts import ChatPromptTemplate
from langchain.chat_models import ChatOpenAI

# 테스트 프롬프트
test_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 데이터 분석 전문가예요. 간결하고 실용적인 답변을 제공해요."),
    ("human", "{question}")
])

# 비교할 모델들 정의
# Playground에서는 UI로 선택하지만 코드에서는 이렇게 설정해요
models_to_compare = {
    "gpt-4o-mini": ChatOpenAI(model="gpt-4o-mini", temperature=0.3),
    # "gpt-4o": ChatOpenAI(model="gpt-4o", temperature=0.3),  # 비용이 더 높아요
}

test_question = "pandas DataFrame에서 중복 행을 제거하는 방법을 설명해줘"

print(f"질문: {test_question}\n")
print("=" * 60)

for model_name, model in models_to_compare.items():
    chain = test_prompt | model
    
    # 실행 시간 측정
    start_time = time.time()
    response = chain.invoke({"question": test_question})
    elapsed = time.time() - start_time
    
    print(f"\n[{model_name}] (실행 시간: {elapsed:.2f}초)")
    print("-" * 40)
    print(response.content)
    print()

질문: pandas DataFrame에서 중복 행을 제거하는 방법을 설명해줘


[gpt-4o-mini] (실행 시간: 4.90초)
----------------------------------------
Pandas DataFrame에서 중복 행을 제거하려면 `drop_duplicates()` 메서드를 사용하면 됩니다. 기본 사용법은 다음과 같습니다:

```python
import pandas as pd

# 예시 DataFrame 생성
data = {
    'A': [1, 2, 2, 3],
    'B': [4, 5, 5, 6]
}
df = pd.DataFrame(data)

# 중복 행 제거
df_unique = df.drop_duplicates()

print(df_unique)
```

### 주요 옵션:
- `subset`: 특정 열을 기준으로 중복을 검사할 때 사용합니다.
- `keep`: 중복된 행 중 어떤 행을 유지할지 설정합니다. ('first', 'last', False)
- `inplace`: 원본 DataFrame에서 직접 수정할지 여부를 결정합니다. (True 또는 False)

예를 들어, 특정 열을 기준으로 중복을 제거하고 싶다면:

```python
df_unique = df.drop_duplicates(subset=['A'], keep='first')
```

이렇게 하면 열 'A'에서 중복된 값이 있는 행 중 첫 번째만 유지됩니다.



## 5. 데이터셋 기반 프롬프트 테스트

단일 입력이 아닌 다양한 케이스를 한 번에 테스트하면 프롬프트의 일반화 성능을 확인할 수 있어요.

### Playground에서 "Test over dataset" 사용하기

1. Playground 하단의 **"Test over dataset"** 드롭다운 클릭
2. 이미 생성된 LangSmith 데이터셋 선택
3. **"Run"** 클릭
4. 각 데이터셋 예제에 대한 응답이 모두 생성돼요
5. 결과를 엑셀처럼 표 형태로 확인 가능해요

> 🎯 **강의 포인트**: 이 기능은 **회귀 테스트**와 같아요. 프롬프트를 수정할 때마다 모든 테스트 케이스를 자동으로 실행해서 의도치 않은 성능 저하를 방지할 수 있어요.

데이터셋을 먼저 만들어볼게요 (Playground 테스트를 위해):

In [7]:
# ---------------------------------------------------
# Playground 테스트용 데이터셋 생성
# ---------------------------------------------------
# LangSmith 클라이언트로 데이터셋을 만들어요
from langsmith import Client

client = Client()

# 데이터셋 이름 (중복 방지)
dataset_name = "customer-service-test-cases"

# 기존 데이터셋이 있으면 삭제 후 재생성
try:
    existing = client.read_dataset(dataset_name=dataset_name)
    client.delete_dataset(dataset_id=existing.id)
    print(f"기존 데이터셋 삭제: {dataset_name}")
except Exception:
    pass

# 새 데이터셋 생성
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="고객 서비스 프롬프트 테스트용 데이터셋"
)

# 테스트 케이스들 (다양한 상황을 커버해요)
test_cases = [
    {
        "inputs": {
            "company_name": "TechShop",
            "max_sentences": "3",
            "customer_inquiry": "주문한 제품이 아직 도착하지 않았어요. 어떻게 해야 하나요?"
        },
        "outputs": {"expected_tone": "empathetic"}  # 참고용 기대값
    },
    {
        "inputs": {
            "company_name": "TechShop",
            "max_sentences": "3",
            "customer_inquiry": "환불 정책이 어떻게 되나요?"
        },
        "outputs": {"expected_tone": "informative"}
    },
    {
        "inputs": {
            "company_name": "TechShop",
            "max_sentences": "3",
            "customer_inquiry": "제품에 결함이 있는 것 같아요. 화가 납니다!"
        },
        "outputs": {"expected_tone": "empathetic_solution"}
    },
]

# 예제들을 데이터셋에 추가
client.create_examples(
    inputs=[tc["inputs"] for tc in test_cases],
    outputs=[tc["outputs"] for tc in test_cases],
    dataset_id=dataset.id
)

print(f"\n데이터셋 생성 완료: {dataset_name}")
print(f"예제 수: {len(test_cases)}개")
print(f"\nLangSmith Playground의 'Test over dataset'에서")
print(f"'{dataset_name}' 데이터셋을 선택하여 테스트해보세요!")

기존 데이터셋 삭제: customer-service-test-cases

데이터셋 생성 완료: customer-service-test-cases
예제 수: 3개

LangSmith Playground의 'Test over dataset'에서
'customer-service-test-cases' 데이터셋을 선택하여 테스트해보세요!


## 6. Polly AI 어시스턴트

Polly는 LangSmith Playground에 내장된 **AI 프롬프트 개선 어시스턴트**예요.
자연어 지시로 프롬프트를 수정하고, 변경 전후를 비교할 수 있어요.

### Polly 사용 방법

1. Playground에서 `Cmd+I` (Mac) 또는 `Ctrl+I` (Windows) 누르기
2. 채팅 사이드바가 열려요
3. 자연어로 수정 요청 입력:

**Polly 활용 예시:**
```
"응답이 너무 길어요. 더 짧고 명확하게 만들어줘"
"공감 표현을 더 많이 추가해줘"
"영어와 한국어를 모두 지원하도록 수정해줘"
"Few-shot 예시를 2개 추가해줘"
```

> 💡 **실무 팁**: Polly가 프롬프트를 수정하면 **Diff 슬라이더**로 변경 전/후를 시각적으로 비교할 수 있어요. 의도하지 않은 부분이 바뀌지 않았는지 꼭 확인해요.

> ⚠️ **자주 하는 실수**: Polly의 제안을 무조건 수용하지 마세요. AI가 제안한 프롬프트도 반드시 테스트해봐야 해요. Polly는 도구이지 정답이 아니에요.

## 7. 프롬프트 저장 및 버전 관리

Playground에서 개선한 프롬프트를 저장하면 자동으로 버전이 관리돼요.

### 저장 및 커밋 프로세스

1. Playground 우상단 **"Save"** 버튼 클릭
2. 프롬프트 이름 입력 (처음 저장 시)
3. 자동으로 **커밋 해시**가 생성돼요 (예: `a3f92c1d`)
4. 이후 수정하고 저장할 때마다 새 커밋이 생성돼요

### 태그로 버전 표시하기

커밋 해시 대신 의미 있는 태그를 붙여서 관리해요:

1. **Prompts** 메뉴 → 저장된 프롬프트 클릭
2. 버전 히스토리에서 특정 커밋 선택
3. **"Add Tag"** 클릭 → `staging` 또는 `production` 입력

```
커밋 히스토리:
  a3f92c1d ← production 태그
  b7d214ef ← staging 태그 (현재 테스트 중)
  c9e8a3b1 (이전 버전)
```

> 🔑 **핵심 개념**: **커밋**은 자동으로 생성되는 불변의 기록이고, **태그**는 사람이 읽기 쉬운 별명이에요. 코드에서 `client.pull_prompt("name:production")`처럼 태그로 특정 버전을 가져올 수 있어요.

In [8]:
# ============================================================
# TODO: Playground에서 개선한 프롬프트를 코드에서 확인하기
# 
# 1. LangSmith Playground에서 고객 서비스 프롬프트를 만들고 저장하세요
#    (이름 예시: "my-customer-service-prompt")
# 2. 아래 코드에서 프롬프트 이름을 수정하고 실행해보세요
# 힌트: client.pull_prompt("이름") 으로 저장된 프롬프트를 가져올 수 있어요
# 예상 결과: Playground에서 저장한 프롬프트가 코드에서 실행돼요
# ============================================================

from langsmith import Client
from langchain.chat_models import ChatOpenAI

client = Client()

# TODO: 여기에 Playground에서 저장한 프롬프트 이름을 입력하세요
PROMPT_NAME = "your-prompt-name"  # ← 실제 이름으로 변경

try:
    # Playground에서 저장한 프롬프트 불러오기
    saved_prompt = client.pull_prompt(PROMPT_NAME)
    print(f"프롬프트 불러오기 성공: {PROMPT_NAME}")
    print(f"프롬프트 구조: {saved_prompt}")
    
    # 모델과 연결하여 실행
    model = ChatOpenAI(model="gpt-4o-mini")
    chain = saved_prompt | model
    
    # 실행 (변수 이름은 프롬프트 구조에 맞게 수정하세요)
    # result = chain.invoke({"변수명": "값"})
    # print(result.content)
    
except Exception as e:
    print(f"프롬프트를 찾을 수 없어요: {e}")
    print("먼저 Playground에서 프롬프트를 저장하고 이름을 수정해주세요.")

프롬프트를 찾을 수 없어요: Resource not found for /commits/-/your-prompt-name/latest. HTTPError('404 Client Error: Not Found for url: https://api.smith.langchain.com/commits/-/your-prompt-name/latest', '{"error":"Commit not found"}\n')
먼저 Playground에서 프롬프트를 저장하고 이름을 수정해주세요.


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **Playground 기본 워크플로우**: 메시지 작성 → 변수 삽입 → 모델 설정 → CMD+Enter 실행 → 저장
- **변수 삽입**: 텍스트에 `{변수명}` 형식으로 작성하거나 선택 후 "Convert to variable" 클릭
- **Trace에서 재실행**: "Open in Playground"로 실제 프로덕션 요청을 가져와 디버깅
- **Side-by-side 비교**: "+" 버튼으로 컬럼 추가하여 여러 모델 동시 비교
- **데이터셋 테스트**: "Test over dataset"으로 여러 케이스를 일괄 실행
- **Polly AI**: `Cmd+I`로 자연어 지시를 통한 프롬프트 자동 개선
- **버전 관리**: 저장 시 자동 커밋 생성 + 수동 태그(`production`, `staging`) 지정

## 다음 노트북 예고

다음 `11-Prompt-Management.ipynb`에서는 **SDK로 프롬프트를 관리하는 방법**을 배워요. `push_prompt()`와 `pull_prompt()`를 활용하여 팀이 협업하는 프롬프트 관리 시스템을 구축해볼게요.